In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, json
import numpy as np
import pandas as pd

FROZEN_DIR = "/content/drive/MyDrive/frozen_dataset"
TRAIN_PATH = f"{FROZEN_DIR}/dataset_v1_train.csv"
TEST_PATH = f"{FROZEN_DIR}/dataset_v1_test.csv"
FULL_PATH = f"{FROZEN_DIR}/dataset_v1_full.csv"

TARGET_COLUMN = "soc_percent"
RAW_FEATURES = ["voltage_v", "current_a", "cell_temp_c", "power_w", "ambient_temp_c"]
CHARGE_TARGET = "remaining_charge_time_s"
CHARGE_FEATURES = ["voltage_v", "current_a", "cell_temp_c", "ambient_temp_c"]

for p in [TRAIN_PATH, TEST_PATH]:
    assert os.path.exists(p), f"Couldn't find {p} — check the Day 12 copy step."

df_train_raw = pd.read_csv(TRAIN_PATH)
df_test_raw = pd.read_csv(TEST_PATH)
train_ids = sorted(df_train_raw["file_id"].unique())
test_ids = sorted(df_test_raw["file_id"].unique())
assert set(train_ids).isdisjoint(test_ids), "Train/test share file_ids - split broken!"
print(f"Frozen split loaded — train: {df_train_raw.shape} ({len(train_ids)} files), test: {df_test_raw.shape} ({len(test_ids)} files)")
if df_train_raw.shape[0] != 76792:
    print("WARNING: Days 15-20 used train (76792, 7) / test (27768, 7). Re-run Day 12 and recopy frozen_dataset before freezing.")

Frozen split loaded — train: (5028, 7) (18 files), test: (328, 7) (5 files)


In [3]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

def make_pipe(estimator, features):
    sel = ColumnTransformer([("keep", "passthrough", features)], remainder="drop")
    return Pipeline([("select", sel), ("model", estimator)])


candidates = {
    "Mean baseline":        (lambda: DummyRegressor(strategy="mean"), 0),
    "Linear (raw)":         (lambda: LinearRegression(), 1),
    "RandomForest (tuned)": (lambda: RandomForestRegressor(n_estimators=300, max_depth=8, min_samples_leaf=4, random_state=42, n_jobs=-1), 2),
    "HistGradBoost (raw)":  (lambda: HistGradientBoostingRegressor(max_iter=300, max_depth=6, learning_rate=0.05, random_state=42), 3),
}

X, y, groups = df_train_raw[RAW_FEATURES], df_train_raw[TARGET_COLUMN], df_train_raw["file_id"]
gkf = GroupKFold(n_splits=min(5, len(train_ids)))
rows = []
for name, (factory, rank) in candidates.items():
    maes, rmses, r2s = [], [], []
    for tr, va in gkf.split(X, y, groups):
        m = make_pipe(factory(), RAW_FEATURES).fit(X.iloc[tr], y.iloc[tr])
        pred = np.clip(m.predict(X.iloc[va]), 0, 100)
        maes.append(mean_absolute_error(y.iloc[va], pred))
        rmses.append(np.sqrt(mean_squared_error(y.iloc[va], pred)))
        r2s.append(r2_score(y.iloc[va], pred))
    rows.append({"model": name, "rank": rank, "cv_MAE": np.mean(maes), "cv_RMSE": np.mean(rmses),
                 "cv_R2": np.mean(r2s), "worst_fold_MAE": np.max(maes)})
    print(f"{name:22s} CV MAE={np.mean(maes):.3f}  RMSE={np.mean(rmses):.3f}  R2={np.mean(r2s):.4f}  worst fold MAE={np.max(maes):.3f}")
cv = pd.DataFrame(rows).set_index("model")
print("\n===== GROUPED CV (train files only) =====")
print(cv.round(4).to_string())

Mean baseline          CV MAE=39.551  RMSE=41.769  R2=-8.5913  worst fold MAE=44.118
Linear (raw)           CV MAE=18.972  RMSE=29.117  R2=-8.7284  worst fold MAE=29.847
RandomForest (tuned)   CV MAE=11.292  RMSE=20.415  R2=-1.3122  worst fold MAE=26.332
HistGradBoost (raw)    CV MAE=10.462  RMSE=19.363  R2=-1.3687  worst fold MAE=24.125

===== GROUPED CV (train files only) =====
                      rank   cv_MAE  cv_RMSE   cv_R2  worst_fold_MAE
model                                                               
Mean baseline            0  39.5507  41.7687 -8.5913         44.1179
Linear (raw)             1  18.9722  29.1168 -8.7284         29.8468
RandomForest (tuned)     2  11.2925  20.4153 -1.3122         26.3320
HistGradBoost (raw)      3  10.4620  19.3630 -1.3687         24.1248


In [4]:
cand = cv.drop("Mean baseline")
best_mae = cand["cv_MAE"].min()
eligible = cand[cand["cv_MAE"] <= 1.10 * best_mae]
chosen = eligible.sort_values(["rank", "cv_RMSE", "worst_fold_MAE"]).index[0]
row = cv.loc[chosen]
robust_ok = row["worst_fold_MAE"] <= 2 * row["cv_MAE"]

print(f"Best CV MAE: {cand['cv_MAE'].idxmin()} ({best_mae:.3f})")
print(f"Eligible (within 10% of best): {list(eligible.index)}")
print(f"FINAL SOC MODEL: {chosen}")
print(f"CV MAE={row.cv_MAE:.3f}  RMSE={row.cv_RMSE:.3f}  R2={row.cv_R2:.3f}  worst fold={row.worst_fold_MAE:.3f}")
print("Robustness check:", "PASS" if robust_ok else "FLAG - discuss with mentor")
if cand["cv_MAE"].idxmin() != cand["cv_RMSE"].idxmin():
    print(f"NOTE: lowest MAE = {cand['cv_MAE'].idxmin()}, lowest RMSE = {cand['cv_RMSE'].idxmin()} - state this trade-off in the rationale.")

Best CV MAE: HistGradBoost (raw) (10.462)
Eligible (within 10% of best): ['RandomForest (tuned)', 'HistGradBoost (raw)']
FINAL SOC MODEL: RandomForest (tuned)
CV MAE=11.292  RMSE=20.415  R2=-1.312  worst fold=26.332
Robustness check: FLAG - discuss with mentor


In [8]:
import joblib, datetime, platform, sklearn
os.makedirs("final_pipeline_v1", exist_ok=True)


final_pipe = make_pipe(candidates[chosen][0](), RAW_FEATURES).fit(df_train_raw[RAW_FEATURES], df_train_raw[TARGET_COLUMN])
joblib.dump(final_pipe, "final_pipeline_v1/soc_final_pipeline.joblib")

charge_frozen = False
if os.path.exists(FULL_PATH):
    df_full = pd.read_csv(FULL_PATH)
    df_full["prog_time_td"] = pd.to_timedelta(df_full["prog_time"], errors="coerce")
    df_full = df_full.sort_values(["file_id", "prog_time_td"]).reset_index(drop=True)
    df_full["status_change"] = ((df_full["status"] != df_full["status"].shift(1)) |
                                (df_full["file_id"] != df_full["file_id"].shift(1)))
    df_full["segment_id"] = df_full["status_change"].cumsum()
    df_charge = df_full[df_full["status"] == "CHA"].copy()
    seg_end = df_charge.groupby("segment_id")["prog_time_td"].transform("max")
    df_charge[CHARGE_TARGET] = (seg_end - df_charge["prog_time_td"]).dt.total_seconds()
    df_charge = df_charge.dropna(subset=[CHARGE_TARGET] + CHARGE_FEATURES)
    charge_train = df_charge[df_charge["file_id"].isin(train_ids)]
    if charge_train["file_id"].nunique() >= 3:
        charge_pipe = make_pipe(RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1), CHARGE_FEATURES)
        charge_pipe.fit(charge_train[CHARGE_FEATURES], charge_train[CHARGE_TARGET])
        joblib.dump(charge_pipe, "final_pipeline_v1/charging_time_final_pipeline.joblib")
        charge_frozen = True
        print(f"Charging-time RF frozen on {len(charge_train)} rows / {charge_train['file_id'].nunique()} train files")

config = {
    "frozen_on": str(datetime.datetime.now()),
    "chosen_model": chosen,
    "estimator": repr(final_pipe.named_steps["model"]),
    "features": RAW_FEATURES,
    "target": TARGET_COLUMN,
    "preprocessing": "column selection by name only; no scaling; predictions clipped to [0,100]",
    "split": "GroupShuffleSplit by file_id, test_size=0.2, random_state=42 (Day 12)",
    "train_shape": list(df_train_raw.shape), "test_shape": list(df_test_raw.shape),
    "train_file_ids": train_ids, "test_file_ids": test_ids,
    "selection_rule": "GroupKFold CV on train files; within 10% of best CV MAE -> simplest; worst fold <= 2x mean",
    "robustness_pass": bool(robust_ok),
    "charging_time_model_frozen": charge_frozen,
    "metrics": ["MAE", "RMSE", "R2"], "seed": 42,
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
}
json.dump(config, open("final_pipeline_v1/frozen_config.json", "w"), indent=2)
cv.to_csv("final_pipeline_v1/cv_results_soc.csv")

rationale = f"""
Final SOC method: {chosen} on 5 raw sensor features, predictions clipped to [0,100].
Evidence: grouped {gkf.n_splits}-fold CV on training files only - CV MAE {row.cv_MAE:.3f}, RMSE {row.cv_RMSE:.3f}, R2 {row.cv_R2:.3f}, worst fold MAE {row.worst_fold_MAE:.3f}.
Rule (fixed before test): within 10% of best CV MAE -> simplest; robustness check worst fold <= 2x mean ({'PASS' if robust_ok else 'FLAG'}).
Rejected: engineered features (Day 13/17, MAE ~23, R2<0); MLP (Day 17, MAE 25.9); mean baseline = reference only.
Frozen: file-level split, feature list, preprocessing, hyper-parameters, seed, metric protocol (MAE/RMSE/R2). Test set first used in Day 23.
Charging time: separate constrained RF (100 trees, depth 10), judged separately - one CC-CV segment per file, low confidence.
Known limitations: low-SOC knee errors; only 25 degC data; RMSE >> MAE means occasional large mistakes.
"""
open("final_pipeline_v1/selection_rationale.txt", "w").write(rationale)
print(rationale)

Charging-time RF frozen on 72 rows / 9 train files

Final SOC method: RandomForest (tuned) on 5 raw sensor features, predictions clipped to [0,100].
Evidence: grouped 5-fold CV on training files only - CV MAE 11.292, RMSE 20.415, R2 -1.312, worst fold MAE 26.332.
Rule (fixed before test): within 10% of best CV MAE -> simplest; robustness check worst fold <= 2x mean (FLAG).
Rejected: engineered features (Day 13/17, MAE ~23, R2<0); MLP (Day 17, MAE 25.9); mean baseline = reference only.
Frozen: file-level split, feature list, preprocessing, hyper-parameters, seed, metric protocol (MAE/RMSE/R2). Test set first used in Day 23.
Charging time: separate constrained RF (100 trees, depth 10), judged separately - one CC-CV segment per file, low confidence.
Known limitations: low-SOC knee errors; only 25 degC data; RMSE >> MAE means occasional large mistakes.



In [6]:
reloaded = joblib.load("final_pipeline_v1/soc_final_pipeline.joblib")
sample = df_train_raw.sample(500, random_state=42)
a = np.clip(reloaded.predict(sample[RAW_FEATURES]), 0, 100)
b = np.clip(final_pipe.predict(sample[RAW_FEATURES]), 0, 100)
assert np.allclose(a, b), "Reloaded pipeline gives different predictions!"
print("Reload check PASSED - identical predictions on 500 rows.")
print("\n✅ Day 22 artifacts saved in 'final_pipeline_v1/':")
for f in sorted(os.listdir("final_pipeline_v1")):
    print(" -", f)

Reload check PASSED - identical predictions on 500 rows.

✅ Day 22 artifacts saved in 'final_pipeline_v1/':
 - charging_time_final_pipeline.joblib
 - cv_results_soc.csv
 - frozen_config.json
 - selection_rationale.txt
 - soc_final_pipeline.joblib


In [7]:
import shutil
shutil.copytree("final_pipeline_v1", "/content/drive/MyDrive/day22/final_pipeline_v1", dirs_exist_ok=True)

'/content/drive/MyDrive/day22/final_pipeline_v1'